In [ ]:
using CairoMakie
using MAGEMin_C
using CSV
using DataFrames
using JLD2
using Accessors

# Load local modules explicitly each time Cell 1 is run.
# This avoids notebook state issues and makes modules available to later cells.
include("Modules/LatticeStrainModels.jl")
include("Modules/LSMUtils.jl")
include("Modules/PartitionCoefficients.jl")
include("Modules/LSMPlotting.jl")

import .LatticeStrainModels
import .LSMUtils
import .PartitionCoefficientsTest
import .LSMPlotting

Using libMAGEMin.dylib from MAGEMin_jll

In [ ]:
function ECX_run_buffered(bulk_0::Vector{Float64}, TE_0::Vector{Float64}, P::Vector{Float64}, T::Vector{Float64}, boff::Vector{Float64})

    """
     Input:

        bulk_0: Vector of initial bulk composition 
                Input format, mol% or wt%, as well as oxides need to be defined outside of this function
        
        TE_0: Vector of initial trace element concentrations in ppm
              Elements and their charges need to be defined outside of this function

        P: Vector of pressures for each step in kbar

        T: Vector of temperatures for each step in °C

        boff: Vector of buffer offsets for each step in log units
              Buffer needs to be defined outside of this function

    Output:

        Out_XY: Vector of MAGEMin outputs from single point minimization for each step

        Out_Sat: Vector of outputs from MAGEMin saturation model for each step

        Out_TE: Vector of dictionaries of trace element concentrations for each step
                Out_TE[Int(step)][String("phase")][Symbol(:element)] gives the concentration of that element in that phase for that step

        Out_Ds: Vector of dictionaries ofpartition coefficients for each step
                Out_Ds[Int(step)][String("phase")][Symbol(:element)] gives the concentration of that element in that phase for that step
                
        Out_EuR: Vector of Eu/R ratio for each step
    
    """

    # Globals defined in other cells; copy into local bindings for safe, fast access
    sys_in_local    = sys_in
    oxides_local    = oxides
    sat_C0s_local   = sat_C0s
    sat_KdsDB_local = sat_KdsDB
    elements_local  = elements
    charges_local   = charges
    buffer_local    = buffer


    # Initalize MAGEMin
    data = Initialize_MAGEMin("ig", buffer=buffer_local, verbose=false)
    rm_list = remove_phases(["ne", "sph", "ky", "sill", "and", "ep", "fper", "chl", ],"ig");


    # Initialize melt fraction
    meltF = 1.0
    # Initialize counter
    np = 0
    nsteps = length(T)


    # Initialize output storage
    Out_XY = Vector{out_struct}(undef, nsteps)
    Out_Sat = Vector{out_TE_struct}(undef, nsteps)


    # Out_TE should be a vector of dictionaries, where each dictionary corresponds to a step. Each step-dictionary is to be structured like Dict("phase" => Dict(:element => value))
    Out_TE = Vector{Dict{String,Dict{Symbol,Float64}}}(undef, nsteps)


    # Similar to Out_TE 
    Out_Ds = Vector{Dict{String,Dict{Symbol,Float64}}}(undef, nsteps)

    Out_EuR = zeros(Float64, nsteps)

    param_dicts = [Dict{Symbol,Float64}() for _ in 1:nsteps]

    while meltF > 0.0 && np < nsteps

        np += 1

        # Reset initial bulk composition -> No melt sequestration
        bulk = deepcopy(bulk_0)
        sys_in_local = sys_in

        Out_TE[np] = Dict{String,Dict{Symbol,Float64}}()
        Out_Ds[np] = Dict{String,Dict{Symbol,Float64}}()

        # If input bulk composition is in wt.%, convert to mol.%
        if sys_in_local == "wt"
            bulk = wt2mol(bulk, oxides_local)
            sys_in_local = "mol"
        end

        # Routine from https://computationalthermodynamics.github.io/MAGEMin_C.jl/dev/MAGEMin_C/saturation_models#E.4-Saturation-models-and-bulk-correction
        tol = 1e-6
        res = 1.0
        n0 = 0.0
        ite = 0

        while res > tol && ite < 32
            #println(bulk)
            out = single_point_minimization(P[np], T[np], data, X=bulk, Xoxides=oxides_local, B=boff[np], sys_in=sys_in_local, name_solvus=true, rm_list=rm_list)
            Out_XY[np] = deepcopy(out) # Store output

            Out_Sat[np] = TE_prediction(out, sat_C0s_local, sat_KdsDB_local, "ig";
                ZrSat_model="CB",
                P2O5Sat_model="Klein26")


            # Reset P_kbar 
            #out = @set out.P_kbar = pkbar
            bulk .-= Out_Sat[np].bulk_cor_mol
            res = abs(n0 - vec_norm(Out_Sat[np].bulk_cor_mol))
            n0 = vec_norm(Out_Sat[np].bulk_cor_mol)

            ite += 1
            if ite == 32
                @warn "Saturation model did not converge in 32 iterations, residual is $res"
            end

        end

        out = Out_XY[np] # Get the output of the single point minimization for this step

        out_sat = Out_Sat[np] # Get the output of the saturation model for this step

        # Reset oxides
        if np == 1
            oxides_local = ["SiO2"; "Al2O3"; "CaO"; "MgO"; "FeO"; "K2O"; "Na2O"; "TiO2"; "O"; "Cr2O3"; "H2O"]
        end


        # If any of the entries in bulk are NaN, set them to the original value from bulk_0
        for i in eachindex(bulk)
            if isnan(bulk[i])
                bulk[i] = bulk_0[i]
            else
                # Nothing to do
            end
        end


        meltF = out.frac_M
        #println("Step $np: Melt Fraction = $meltF")
        # Break loop if melt fraction is zero or negative (which can happen due to numerical issues)
        if meltF <= 0.0
            println("Melt fraction is zero or negative, stopping iteration.")
            break
        end
        meltF_wt = out.frac_M_wt

        outte = Dict{String,Dict{Symbol,Float64}}()
        out_d = Dict{String,Dict{Symbol,Float64}}()

        phases = copy(out.ph)

        # Work on a local copy of phase fractions so the output `out` is unchanged
        phase_fracs_wt = copy(out.ph_frac_wt)

        # Add 'zr' and 'ap' to phases if zrc_wt > 0 and fapt_wt > 0, respectively
        # Append fractions to the local `phase_fracs_wt` to keep alignment
        if out_sat.zrc_wt > 0.0 && !("zr" in phases)
            push!(phases, "zr")
            push!(phase_fracs_wt, out_sat.zrc_wt)
        end
        if out_sat.fapt_wt > 0.0 && !("ap" in phases)
            push!(phases, "ap")
            push!(phase_fracs_wt, out_sat.fapt_wt)
        end

        for phase in phases
            params = LSMUtils.get_params_for_phase(phase, out, out_sat, T[np], P[np])
            # Skip buffer and melt
            if phase in ["liq", buffer]
                out_d[phase] = Dict(el => 0.0 for el in elements)
            else
                Dphase = PartitionCoefficientsV2.get_D(phase, elements, charges, params)
                out_d[phase] = Dphase
            end
        end

        params = LSMUtils.get_params_for_phase("liq", out, out_sat, T[np], P[np])

        Out_EuR[np] = LSMUtils.Eu_ratio_Burnham(params[:logfO2], params[:T], params[:Λ])

        # Need to normalize phase_fracs_wt to 1 while omitting "liq" and "buffer" fractions, it can be done for all fractions as "liq" and "buffer" have zero Kds and thus do not contribute to the sum
        if "liq" in phases
            # I want to normalize phase fractions such that the sum of all fractions excluding "liq" is 1.0, "liq" fraction can also be modified, it won't have any effect as Kd is 0
            phase_fracs_wt_used = phase_fracs_wt ./ (1.0 - phase_fracs_wt[findfirst(==("liq"), phases)])
            #=if mod(np, 10) == 0
                println("Step $np: Normalized phase fractions (excluding 'liq') = $phase_fracs_wt_used")
            end=#
        else
            phase_fracs_wt_used = phase_fracs_wt ./ sum(phase_fracs_wt)
        end

        has_other_phases = any(phase -> !(phase in ["liq", buffer]), phases)

        if has_other_phases
            Kd = Dict(el => sum((phase_fracs_wt_used[i] * get(out_d[phases[i]], el, 0.0) for i in eachindex(phases)); init=0.0) for el in elements)

            # Add Kd to out_d with key "bulk"
            out_d["bulk"] = Kd

            Out_TE[np]["liq"] = Dict(el => TE_0[i] / (Kd[el] + meltF_wt * (1.0 - Kd[el])) for (i, el) in enumerate(elements))
        else
            out_d["bulk"] = Dict(el => NaN for el in elements)

            Out_TE[np]["liq"] = Dict(el => TE_0[i] for (i, el) in enumerate(elements))
        end

        for phase in phases
            # Skip buffer and melt
            if phase in ["liq", buffer]
                continue
            else
                # Compute trace element concentrations in this phase
                Out_TE[np][phase] = Dict(el => Out_TE[np]["liq"][el] * get(out_d[phase], el, 0.0) for el in elements)
            end
        end

        Out_Ds[np] = out_d

    end

    return Out_XY[1:np], Out_Sat[1:np], Out_TE[1:np], Out_Ds[1:np], Out_EuR[1:np], param_dicts[1:np]

end

ECX_run_buffered (generic function with 1 method)

In [ ]:
function FCX_run_buffered(bulk_0::Vector{Float64}, TE_0::Vector{Float64}, P::Vector{Float64}, T::Vector{Float64}, boff::Vector{Float64})


    """
     Input:

        bulk_0: Vector of initial bulk composition 
                Input format, mol% or wt%, as well as oxides need to be defined outside of this function
        
        TE_0: Vector of initial trace element concentrations in ppm
              Elements and their charges need to be defined outside of this function

        P: Vector of pressures for each step in kbar

        T: Vector of temperatures for each step in °C

        boff: Vector of buffer offsets for each step in log units
              Buffer needs to be defined outside of this function

    Output:

        Out_XY: Vector of MAGEMin outputs from single point minimization for each step

        Out_Sat: Vector of outputs from MAGEMin saturation model for each step

        Out_TE: Vector of dictionaries of trace element concentrations for each step
                Out_TE[Int(step)][String("phase")][Symbol(:element)] gives the concentration of that element in that phase for that step

        Out_Ds: Vector of dictionaries ofpartition coefficients for each step
                Out_Ds[Int(step)][String("phase")][Symbol(:element)] gives the concentration of that element in that phase for that step
                
        Out_EuR: Vector of Eu/R ratio for each step
    
    """


    # Globals defined in other cells; copy into local bindings for safe, fast access
    sys_in_local    = sys_in
    oxides_local    = oxides
    sat_C0s_local   = sat_C0s
    sat_KdsDB_local = sat_KdsDB
    elements_local  = elements
    charges_local   = charges
    buffer_local    = buffer


    # Initalize MAGEMin
    data = Initialize_MAGEMin("ig", buffer=buffer_local, verbose=false)
    rm_list = remove_phases(["ne", "sph", "ky", "sill", "and", "ep", "fper", "chl", ],"ig");


    # Initialize melt fraction
    meltF = 1.0
    # Initialize counter
    np = 0
    nsteps = length(T)

    # Copy bulk_0 once for initialiazation
    bulk = deepcopy(bulk_0)
    sys_in_local = sys_in


    # Initialize output storage
    Out_XY = Vector{out_struct}(undef, nsteps)
    Out_Sat = Vector{out_TE_struct}(undef, nsteps)


    # Out_TE should be a vector of dictionaries, where each dictionary corresponds to a step. Each step-dictionary is to be structured like Dict("phase" => Dict(:element => value))
    Out_TE = Vector{Dict{String,Dict{Symbol,Float64}}}(undef, nsteps)


    # Similar to Out_TE 
    Out_Ds = Vector{Dict{String,Dict{Symbol,Float64}}}(undef, nsteps)

    Out_EuR = zeros(Float64, nsteps)

    param_dicts = [Dict{Symbol,Float64}() for _ in 1:nsteps]

    while meltF > 0.0 && np < nsteps

        np += 1

        Out_TE[np] = Dict{String,Dict{Symbol,Float64}}()
        Out_Ds[np] = Dict{String,Dict{Symbol,Float64}}()

        # If input bulk composition is in wt.%, convert to mol.%
        if sys_in_local == "wt"
            bulk = wt2mol(bulk, oxides_local)
            sys_in_local = "mol"
        end

        # Routine from https://computationalthermodynamics.github.io/MAGEMin_C.jl/dev/MAGEMin_C/saturation_models#E.4-Saturation-models-and-bulk-correction
        tol = 1e-6
        res = 1.0
        n0 = 0.0
        ite = 0

        while res > tol && ite < 32
            #println(bulk)
            out = single_point_minimization(P[np], T[np], data, X=bulk, Xoxides=oxides_local, B=boff[np], sys_in=sys_in_local, name_solvus=true, rm_list=rm_list)
            Out_XY[np] = deepcopy(out) # Store output

            Out_Sat[np] = TE_prediction(out, sat_C0s_local, sat_KdsDB_local, "ig";
                ZrSat_model="CB",
                P2O5Sat_model="Klein26")


            # Reset P_kbar 
            #out = @set out.P_kbar = pkbar
            bulk .-= Out_Sat[np].bulk_cor_mol
            res = abs(n0 - vec_norm(Out_Sat[np].bulk_cor_mol))
            n0 = vec_norm(Out_Sat[np].bulk_cor_mol)

            ite += 1
            if ite == 32
                @warn "Saturation model did not converge in 32 iterations, residual is $res"
            end

        end

        out = Out_XY[np] # Get the output of the single point minimization for this step

        out_sat = Out_Sat[np] # Get the output of the saturation model for this step

        # Reset oxides
        if np == 1
            oxides_local = ["SiO2"; "Al2O3"; "CaO"; "MgO"; "FeO"; "K2O"; "Na2O"; "TiO2"; "O"; "Cr2O3"; "H2O"]
        end

        # Update bulk for next iteration
        bulk = out.bulk_M


        # If any of the entries in bulk are NaN, set them to the original value from bulk_0
        for i in eachindex(bulk)
            if isnan(bulk[i])
                bulk[i] = bulk_0[i]
            else
                # Nothing to do
            end
        end


        meltF = out.frac_M
        #println("Step $np: Melt Fraction = $meltF")
        # Break loop if melt fraction is zero or negative (which can happen due to numerical issues)
        if meltF <= 0.0
            println("Melt fraction is zero or negative, stopping iteration.")
            break
        end
        meltF_wt = out.frac_M_wt

        outte = Dict{String,Dict{Symbol,Float64}}()
        out_d = Dict{String,Dict{Symbol,Float64}}()

        phases = copy(out.ph)

        # Work on a local copy of phase fractions so the output `out` is unchanged
        phase_fracs_wt = copy(out.ph_frac_wt)

        # Add 'zr' and 'ap' to phases if zrc_wt > 0 and fapt_wt > 0, respectively
        # Append fractions to the local `phase_fracs_wt` to keep alignment
        if out_sat.zrc_wt > 0.0 && !("zr" in phases)
            push!(phases, "zr")
            push!(phase_fracs_wt, out_sat.zrc_wt)
        end
        if out_sat.fapt_wt > 0.0 && !("ap" in phases)
            push!(phases, "ap")
            push!(phase_fracs_wt, out_sat.fapt_wt)
        end

        for phase in phases
            params = LSMUtils.get_params_for_phase(phase, out, out_sat, T[np], P[np])
            # Skip buffer and melt
            if phase in ["liq", buffer]
                out_d[phase] = Dict(el => 0.0 for el in elements)
            else
                Dphase = PartitionCoefficientsV2.get_D(phase, elements, charges, params)
                out_d[phase] = Dphase
                #Out_TE[np][phase] = Dict(el => TE_now[i] * Dphase[el] for (i, el) in enumerate(elements))
                if np==1
                    Out_TE[np][phase] = Dict(el => TE_0[i] * Dphase[el] for (i, el) in enumerate(elements))
                else
                    Out_TE[np][phase] = Dict(el => Out_TE[np-1]["liq"][el] * Dphase[el] for (i, el) in enumerate(elements))
                end
            end
        end

        params = LSMUtils.get_params_for_phase("liq", out, out_sat, T[np], P[np])

        Out_EuR[np] = LSMUtils.Eu_ratio_Burnham(params[:logfO2], params[:T], params[:Λ])

        # Need to normalize phase_fracs_wt to 1 while omitting "liq" and "buffer" fractions, it can be done for all fractions as "liq" and "buffer" have zero Kds and thus do not contribute to the sum
        if "liq" in phases
            # I want to normalize phase fractions such that the sum of all fractions excluding "liq" is 1.0, "liq" fraction can also be modified, it won't have any effect as Kd is 0
            phase_fracs_wt_used = phase_fracs_wt ./ (1.0 - phase_fracs_wt[findfirst(==("liq"), phases)])
            #=if mod(np, 10) == 0
                println("Step $np: Normalized phase fractions (excluding 'liq') = $phase_fracs_wt_used")
            end=#
        else
            phase_fracs_wt_used = phase_fracs_wt ./ sum(phase_fracs_wt)
        end
        has_other_phases = any(ph -> !(ph in ["liq", buffer]), phases)

        if has_other_phases
            Kd = Dict(el => sum((phase_fracs_wt_used[i] * get(out_d[phases[i]], el, 0.0) for i in eachindex(phases)); init=0.0) for el in elements)

            # Add Kd to out_d with key "bulk"
            out_d["bulk"] = Kd

            if np == 1
                Out_TE[np]["liq"] = Dict(el => TE_0[i] / (Kd[el] + meltF_wt * (1.0 - Kd[el])) for (i, el) in enumerate(elements))
            else
                Out_TE[np]["liq"] = Dict(el => Out_TE[np-1]["liq"][el] / (Kd[el] + meltF_wt * (1.0 - Kd[el])) for (i, el) in enumerate(elements))
            end
        else
            Kd = Dict(el => NaN for el in elements)

            # Add Kd to out_d with key "bulk"
            out_d["bulk"] = Kd

            if np==1
                Out_TE[np]["liq"] = Dict(el => TE_0[i] for (i, el) in enumerate(elements))
            else
                Out_TE[np]["liq"] = Dict(el => Out_TE[np-1]["liq"][el] for (i, el) in enumerate(elements))
            end
        end

        Out_Ds[np] = out_d

    end

    return Out_XY[1:np], Out_Sat[1:np], Out_TE[1:np], Out_Ds[1:np], Out_EuR[1:np], param_dicts[1:np]

end

FCX_run_buffered (generic function with 1 method)

In [ ]:
# Bulk composition dictionary "Rock name" => [oxide bulk composition in wt.%], anhydrous!

"""
    Reference:

    Schmidt, M. W., and O. Jagoutz (2017), The global systematics of primitive arc melts, Geochem. Geophys. Geosyst., 18, 2817–2854, doi:10.1002/ 2016GC006699.

"""

oxides = ["SiO2"; "Al2O3"; "CaO"; "MgO"; "FeO"; "K2O"; "Na2O"; "TiO2"; "O"; "Cr2O3"; "H2O"]
sys_in = "wt"

bulk_input_dict = Dict(
    "Cascades_Thol_Basalt" => [48.6, 17.7, 11.54, 10.03, 8.55, 0.16, 2.45, 0.71, 0.0, 279e-4, 0.0],
    "Cascades_CA_Basalt" => [50.6, 16.3, 9.35, 9.09, 8.25, 1.21, 3.29, 1.30, 0.0, 328e-4, 0.0],
    "Mexico_CA_Basalt" => [50.7, 16.0, 9.10, 9.51, 7.98, 1.49, 3.55, 1.04, 0.0, 452e-4, 0.0],
    "Mexico_LowSi_Basalt" => [45.8, 14.1, 10.53, 13.47, 10.37, 0.93, 2.61, 1.49, 0.0, 853e-4, 0.0],
    "Mexico_Shoshonitic" => [51.3, 13.3, 8.03, 9.56, 7.27, 4.51, 3.14, 1.57, 0.0, 412e-4, 0.0],
    "Vanuatu_CA_Basalt" => [49.2, 12.9, 11.21, 12.93, 9.69, 0.97, 1.99, 0.69, 0.0, 687e-4, 0.0],
    "Palau_Thol_Basalt" => [50.9, 16.2, 10.52, 10.93, 8.03, 0.26, 2.38, 0.57, 0.0, 658e-4, 0.0]
)

elements = [:La, :Ce, :Pr, :Nd, :Sm, :Eu, :Gd, :Tb, :Dy, :Ho, :Er, :Tm, :Yb, :Lu, :Sr, :Y]
charges = [3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3]

TE_input_dict = Dict(
    "Cascades_Thol_Basalt" => [3.1, 7.5, 1.2, 5.9, 1.9 , 0.8, 2.5, 0.41, 3.3, 0.63, 2.2, 0.28, 2.0, 0.30, 231.0, 19.7],
    "Cascades_CA_Basalt" => [22.9, 49.4, 7.5, 25.4, 5.2, 1.6, 4.6, 0.60, 3.8, 0.71, 2.0, 0.28, 1.7, 0.26, 999.0, 20.4],
    "Mexico_CA_Basalt" => [27.2, 55.5, 6.1, 22.5, 4.3, 1.3, 3.8, 0.57, 3.2, 0.69, 1.7, 0.38, 1.8, 0.25, 1012.0, 17.9],
    "Mexico_LowSi_Basalt" => [33.1, 62.6, 7.9, 29.5, 5.8, 1.9, 5.5, 0.67, 4.0, 0.88, 2.3, 0.31, 1.5, 0.22, 675.0, 22.3],
    "Mexico_Shoshonitic" => [54.4, 117.9, 16.1, 60.4, 10.7, 2.9, 7.8, 0.89, 4.2, 0.77, 2.0, 0.26, 1.6, 0.20, 2600.0, 21.6],
    "Vanuatu_CA_Basalt" => [10.1, 22.1, 2.9, 13.6, 2.8, 0.9, 2.9, 0.45, 2.7, 0.55, 1.7, 0.25, 1.6, 0.27, 548.0, 16.4],
    "Palau_Thol_Basalt" => [2.4, 6.2, 0.9, 5.0, 1.5, 0.6, 2.0, 0.36, 2.4, 0.49, 1.4, 0.22, 1.4, 0.22, 147.0, 13.5]
 )

sat_input_dict = Dict(
    "Cascades_Thol_Basalt" => [57.0, 0.08e4],
    "Cascades_CA_Basalt" => [152.0, 0.42e-4],
    "Mexico_CA_Basalt" => [137.0, 0.44e4],
    "Mexico_LowSi_Basalt" => [155.0, 0.46e4],
    "Mexico_Shoshonitic" => [454.0, 0.89e4],
    "Vanuatu_CA_Basalt" => [49.0, 0.20e4],
    "Palau_Thol_Basalt" => [40.0, 0.06e4]
)





In [ ]:

# For zircon and apatite saturation models
sat_elements = ["Zr", "P2O5"]
#sat_phases = ["zrc", "fapt"]
sat_phases = ["cpx", "pig", "Na-cpx", "opx", "pl", "afs", "amp", "gl", "act", "cumm", "tr", "bi", "g", "ilm", "hem", "ep", "ru", "spl", "cm", "usp", "mgt", "zr", "ap"]
# Fixed Kds from Bédard 2006; DOI:10.1016/j.gca.2005.11.008
sat_Kds = [ "0.125" "0.162";
            "0.125" "0.162";
            "0.125" "0.162";
            "0.031" "0.050";
            "0.078" "0.079";
            "0.078" "0.079";
            "0.417" "0.225";
            "0.417" "0.225";
            "0.417" "0.225";
            "0.417" "0.225";
            "0.417" "0.225";
            "0.023" "0.005";
            "0.537" "0.184";
            "2.3" "0.002";
            "2.3" "0.002";
            "0.10" "0.18";
            "3.7" "0.03";
            "0.12" "0.024";
            "0.12" "0.024";
            "0.12" "0.024";
            "0.12" "0.024";
            "0.0" "20.0";
            "16.0" "0.0"]

sat_KdsDB = create_custom_KDs_database(sat_elements, sat_phases, sat_Kds)

# sat_C0s = [96.0,2200.0] => Set from sat_input_dict according to selected bulk composition

sat_models = ["CB", "Klein26"]

2-element Vector{String}:
 "CB"
 "Klein26"

In [ ]:
pressures = [4.0, 7.0, 9.0]

h2os = [2.0, 4.0, 6.0, 8.0]

#h2o = 2.0

buffer = "qfm"

dfo2s = [0.0, 1.0]

T = collect(1300.0:-5.0:600.0)

#boffs = fill(boff, length(T))

141-element Vector{Float64}:
 1300.0
 1295.0
 1290.0
 1285.0
 1280.0
 1275.0
 1270.0
 1265.0
 1260.0
 1255.0
 1250.0
 1245.0
 1240.0
    ⋮
  655.0
  650.0
  645.0
  640.0
  635.0
  630.0
  625.0
  620.0
  615.0
  610.0
  605.0
  600.0

In [ ]:
# Run ECX and FCX for different pressures, h2o contents, and bulk compositions
# For each bulk composition set the last entry to h2o and multiply all other entries by (1.0 - h2o/100.0) to account for the addition of water

to_run = ["Cascades_CA_Basalt", "Cascades_Thol_Basalt", "Mexico_CA_Basalt", "Vanuatu_CA_Basalt"]

for (bulk_name, bulk_wt) in bulk_input_dict
    if !(bulk_name in to_run)
        continue
    end
    TE_0 = TE_input_dict[bulk_name]
    global sat_C0s = sat_input_dict[bulk_name]


    for pr in pressures

        P = fill(pr, length(T))

        for h2o in h2os

                bulk_wt_hydr = deepcopy(bulk_wt)
                bulk_wt_hydr = bulk_wt_hydr ./ sum(bulk_wt_hydr) .* 100.0 # Normalize to 100 wt.% before adding water
                bulk_wt_hydr[end] = h2o  # Set H2O content
                bulk_wt_hydr[1:end-1] .*= (1.0 - h2o / 100.0)  # Scale other oxides
                bulk_wt_hydr[end-2] = 2.0 # For buffering

            for boff in dfo2s

                boffs = fill(boff, length(T))
                

                println("Running ECX for $bulk_name at $pr kbar, H2O = $h2o wt.%, dQFM = $boff log units")
                filepath = "Data/Outputs/SJ17/ECX_$(bulk_name)_$(Int(pr))kbar_$(Int(h2o))H2O_$(Int(boff))dQFM.jld2"
                if isfile(filepath)
                    println("File $filepath already exists. Skipping to next iteration.")
                else
                    Out_XY, Out_Sat, Out_TE, Out_Ds, Out_EuR, param_dicts = ECX_run_buffered(bulk_wt_hydr, TE_0, P, T, boffs)
                    # If this file already exists, continue to the next iteration to avoid overwriting
                    @save filepath Out_XY Out_Sat Out_TE Out_Ds Out_EuR param_dicts
                end

                println("Running FCX for $bulk_name at $pr kbar, H2O = $h2o wt.%, dQFM = $boff log units")
                filepath_FCX = "Data/Outputs/SJ17/FCX_$(bulk_name)_$(Int(pr))kbar_$(Int(h2o))H2O_$(Int(boff))dQFM.jld2"
                if isfile(filepath_FCX)
                    println("File $filepath_FCX already exists. Skipping to next iteration.")
                else
                    Out_XY, Out_Sat, Out_TE, Out_Ds, Out_EuR, param_dicts = FCX_run_buffered(bulk_wt_hydr, TE_0, P, T, boffs)
                    @save filepath_FCX Out_XY Out_Sat Out_TE Out_Ds Out_EuR param_dicts
                end
            end
        end
    end

end

In [ ]:
# For each jld2 file in the SJ17 folder get out.ph and out.ph_frac_wt for out in Out_XY and construct a DataFrame with columns
# Step, Temperature [°C], Canonical Phase for all phases that appeared in all out.phs (several columns)
# Step => Index of out in Out_XY
# Temperature [°C] => out.T_C
# Phase => Phase fraction in weight fraction for that phase, contained in out.ph_frac_wt accounting for aliased phases

# Need to collect all aliased phases into the canonical ones in terms of weight fraction. For example, if "Na-cpx" and "pig" are present, they should be summed into "cpx"
# There might be the case where the canonical phase is also present, in which case it should also be summed into the canonical phase. For example, if "cpx" is present along with "Na-cpx" and "pig", all three should be summed into "cpx"
# Some phases might only be canonical too

# If a new phase appears then it should be added to the DataFrame as a new column. All previous steps then have NaN phase fractions
# If a phase disappears all following steps should have NaN phase fractions for that phase

PHASE_ALIASES = Dict{String,String}(
    "Na-cpx" => "cpx",
    "pig"    => "cpx",
    "pl"     => "fsp",
    "afs"    => "fsp",
    "hem"    => "ilm",
    "gl"     => "amp",
    "act"    => "amp",
    "cumm"   => "amp",
    "tr"     => "amp",
    "cm"     => "spl",
    "usp"    => "spl",
    "mgt"    => "spl",
	"pat"	 => "mu",
	"sill"   => "ky",
	"and"    => "ky"
 )

# Get all file names in the SJ17 folder that end with .jld2
jld2_files = filter(endswith(".jld2"), readdir("Data/Outputs/SJ17/"))

# Dict of dataframes for each file, filename without .jld2 => DataFrame

df_dict = Dict{String,DataFrame}()

for file in jld2_files
    filepath = joinpath("Data/Outputs/SJ17/", file)
    @load filepath Out_XY Out_Sat

    df_key = splitext(file)[1]  # Get filename without extension

    steps = collect(1:length(Out_XY))
    temperatures = [out.T_C for out in Out_XY]

    # First pass: gather all canonical phases that appear in this file.
    all_phases = String[]
    for out in Out_XY
        for ph in out.ph
            canonical_phase = get(PHASE_ALIASES, ph, ph)
            if !(canonical_phase in all_phases)
                push!(all_phases, canonical_phase)
            end
        end
    end

    # Add 'zr' and 'ap' to all_phases in any case
    if !("zr" in all_phases)
        push!(all_phases, "zr")
    end
    if !("ap" in all_phases)
        push!(all_phases, "ap")
    end

    # Build a wide table directly: one row per step, one column per canonical phase.
    df = DataFrame(Step=steps, Temperature_C=temperatures)
    for ph in all_phases
        df[!, ph] = fill(NaN, length(Out_XY))
    end

    for (row_idx, out) in enumerate(Out_XY)
        phase_frac_dict = Dict{String,Float64}()

        # Sum aliased phases into their canonical names for this step.
        for (ph, frac) in zip(out.ph, out.ph_frac_wt)
            canonical_phase = get(PHASE_ALIASES, ph, ph)
            phase_frac_dict[canonical_phase] = get(phase_frac_dict, canonical_phase, 0.0) + frac
        end

        # Fill only the phases present at this step; missing ones stay NaN.
        for (ph, frac) in phase_frac_dict
            df[row_idx, ph] = frac
        end
        
        out_sat = Out_Sat[row_idx]  # Get the corresponding saturation output for this step
        # Add 'zr' and 'ap' to phases if out_sat.zrc_wt > 0 and out_sat.fapt_wt > 0, respectively
        # Add corresponding fractions to the DataFrame
        if out_sat.zrc_wt > 0.0
            df[row_idx, "zr"] = out_sat.zrc_wt
        end
        if out_sat.fapt_wt > 0.0
            df[row_idx, "ap"] = out_sat.fapt_wt
        end
    end

    df_dict[df_key] = df

end

In [ ]:
# Save df_dict to a JLD2 file

@save "Data/Outputs/SJ17/DataFrames/PhaseFractions_SJ17.jld2" df_dict

In [ ]:
# Iterate over all file names in "Data/Outputs/SJ17" while ignoring the "/DataFrames" subfolder

filenames = filter(f -> !occursin("DataFrames", f), readdir("Data/Outputs/SJ17/"))

elements = [:La, :Ce, :Pr, :Nd, :Sm, :Eu, :Gd, :Tb, :Dy, :Ho, :Er, :Tm, :Yb, :Lu, :Sr, :Y]

# Convert elements to strings for DataFrame column names
element_strs = string.(elements)

for file in filenames
    # Load Out_XY and Out_TE from this file
    filepath = joinpath("Data/Outputs/SJ17/", file)
    @load filepath Out_XY Out_TE
    # Create a DataFrame that has columns: Step, Temperature [°C], Oxides for all oxides in out.oxides, Elements for all elements in elements
    # Step => Index of out in Out_XY
    # Temperature [°C] => out.T_C
    # Oxides => out.bulk_M_wt for each oxide in out.oxides
    # Elements => Out_TE[Int(step)][String("liq")][Symbol(:element)] for each element in elements

    # Define cutoff indeces
    # Case 1) "ECX" in file name => cutoff index = index where out.frac_M_wt < 1e-3
    # Case 2) "FCX" in file name => cutoff index = index where out.frac_M_wt < 0.5
    cutoff_index = nothing
    if occursin("ECX", file)
        for (i, out) in enumerate(Out_XY)
            if out.frac_M_wt < 0.01
                cutoff_index = i
                break
            end
        end
    elseif occursin("FCX", file)
        for (i, out) in enumerate(Out_XY)
            if out.frac_M_wt < 0.5
                cutoff_index = i
                break
            end
        end
    end

    if cutoff_index !== nothing
        Out_XY = Out_XY[1:cutoff_index-1]
        Out_TE = Out_TE[1:cutoff_index-1]
    end


    steps = collect(1:length(Out_XY))
    temperatures = [out.T_C for out in Out_XY]

    # Initialize DataFrame with Step and Temperature columns
    df = DataFrame(Step=steps, Temperature_C=temperatures)

    # Add oxide columns
    for (i, oxide) in enumerate(Out_XY[1].oxides)
        df[!, oxide] = [out.bulk_M_wt[i] for out in Out_XY]
    end

    # Add element columns
    for (i, element) in enumerate(elements)
        df[!, string(element)] = [Out_TE[step]["liq"][element] for step in steps]
    end

    # Save to a CSV file of the same filename in "Data/Outputs/SJ17/DataFrames" folder, with the same name as the original file but with .csv extension
    csv_filename = replace(file, ".jld2" => ".csv")
    CSV.write(joinpath("Data/Outputs/SJ17/DataFrames", csv_filename), df)
end